In [10]:
import pprint

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon

In [11]:

world_filepath = gpd.datasets.get_path('naturalearth_lowres')
world = gpd.read_file(world_filepath)

/tmp/ipykernel_347154/947498767.py:1: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world_filepath = gpd.datasets.get_path('naturalearth_lowres')


In [12]:
min_lon = -180
max_lon = 180
min_lat = -30
max_lat = 30
tropics = Polygon([(min_lon, min_lat), (min_lon, max_lat), (max_lon, max_lat, ), (max_lon, min_lat )])


In [13]:
tropical_countries = world.loc[world.intersects(tropics)]

In [14]:
#Equal-area projection
tropical_countries['area'] = \
    tropical_countries.to_crs('EPSG:6933').area / 1000000

/home/jan/miniconda3/envs/torch/lib/python3.11/site-packages/geopandas/geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [15]:
tropical_countries

,pop_est,continent,name,iso_a3,gdp_md_est,geometry,area
0,889953.0,Oceania,Fiji,FJI,5496,"MULTIPOLYGON (((180.00000 -16.06713, 180.00000...",1.928760e+04
1,58005463.0,Africa,Tanzania,TZA,63177,"POLYGON ((33.90371 -0.95000, 34.07262 -1.05982...",9.327793e+05
2,603253.0,Africa,W. Sahara,ESH,907,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948...",9.666925e+04
4,328239523.0,North America,United States of America,USA,21433226,"MULTIPOLYGON (((-122.84000 49.00000, -120.0000...",9.509851e+06
7,8776109.0,Oceania,Papua New Guinea,PNG,24829,"MULTIPOLYGON (((141.00021 -2.60015, 142.73525 ...",4.645316e+05
...,...,...,...,...,...,...,...
167,5096159.0,Africa,Somaliland,SOL,17836,"POLYGON ((48.94820 11.41062, 48.94820 11.41062...",1.674732e+05
168,44269594.0,Africa,Uganda,UGA,35165,"POLYGON ((33.90371 -0.95000, 31.86617 -1.02736...",2.457603e+05
169,12626950.0,Africa,Rwanda,RWA,10354,"POLYGON ((30.41910 -1.13466, 30.81613 -1.69891...",2.336524e+04
175,1394973.0,North America,Trinidad and Tobago,TTO,24269,"POLYGON ((-61.68000 10.76000, -61.10500 10.890...",7.738923e+03


In [16]:
df = pd.read_csv('../data/tmp/FAOSTAT_data_en_4-1-2024.csv')

In [90]:
df.Item.unique()

array(['Cattle', 'Soya beans', 'Cocoa beans', 'Coffee, green', 'Palm oil',
       'Natural rubber in primary forms'], dtype=object)

In [91]:
#example
#Here the number is negative - 
#ie. area harvested is smaller in 2020 than in 2010; 
#therefore no expansion and i should enter 0 in the model
#cocoa_fiji = _df.loc[(_df.Element == 'Area harvested') & ( _df.Area == 'Fiji') & ( _df.Item == 'Cocoa beans')]
#harvested_2010 = cocoa_fiji.loc[cocoa_fiji.Year == 2010].Value.values[0] 
#harvested_2020 = cocoa_fiji.loc[cocoa_fiji.Year == 2020].Value.values[0] 


In [19]:
df.loc[(df.Element == 'Area harvested') & ( df.Area == 'Malaysia') & ( df.Item == 'Natural rubber in primary forms')]







,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
1244,QCL,Crops and livestock products,458,Malaysia,5312,Area harvested,1950.01,Natural rubber in primary forms,2000,2000,ha,1430680.0,A,Official figure,NaN
1247,QCL,Crops and livestock products,458,Malaysia,5312,Area harvested,1950.01,Natural rubber in primary forms,2020,2020,ha,1139140.0,A,Official figure,NaN


In [92]:
FAOSTAT_classes = ['Cattle', 'Palm oil', 'Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms']

In [93]:
country_name_mapping = {
    "Vietnam" : "Viet Nam",
    "Tanzania" : "United Republic of Tanzania",
    "Dem. Rep. Congo" : "Democratic Republic of the Congo",
    "Dominican Rep." : "Dominican Republic",
    "Bolivia" : "Bolivia (Plurinational State of)",
    "Venezuela" : "Venezuela (Bolivarian Republic of)",
    "Central African Rep.": "Central African Republic",
    "Eq. Guinea" : "Equatorial Guinea",
    "eSwatini" : "Eswatini",
    "Laos" : "Lao People's Democratic Republic",
    "Taiwan" : "China, Taiwan Province of",
    "Brunei" : "Brunei Darussalam",
    "S. Sudan" : "South Sudan",
    "Solomon Is." : "Solomon Islands"
}

In [94]:
names_to_edit = [key for key in country_name_mapping.keys()]
for name in names_to_edit:
    tropical_countries.loc[tropical_countries['name'] == name, 'name'] = country_name_mapping[name]
    

In [95]:
elements = {
    'Cattle' : {}, 
    'Palm oil' : {}, 
    'Soya beans' : {}, 
    'Cocoa beans' : {}, 
    'Coffee, green' : {}, 
    'Natural rubber in primary forms' : {}
}

for idx, row in tropical_countries.iterrows():
    
    #try:
    #    country_name = country_name_mapping[row['name']]
    #except KeyError:
    country_name = row['name']
    
    _df = df.loc[df.Area == country_name]
    if len(_df) == 0:
        print('No FAOSTAT data found for ', row['name'])
    for _cls in FAOSTAT_classes:
        commodity_df = _df.loc[_df.Item == _cls]
        elements_available = commodity_df.Element.unique()
        
        for element in elements_available:
            try:
                elements[_cls][element] += 1
            except KeyError:
                elements[_cls][element] = 1

No FAOSTAT data found for  W. Sahara
No FAOSTAT data found for  Iran
No FAOSTAT data found for  Somaliland


In [96]:
pprint.pprint(elements)


{'Cattle': {'Stocks': 110},
 'Cocoa beans': {'Area harvested': 52, 'Production': 52, 'Yield': 50},
 'Coffee, green': {'Area harvested': 70, 'Production': 70, 'Yield': 69},
 'Natural rubber in primary forms': {'Area harvested': 35,
                                     'Production': 35,
                                     'Yield': 31},
 'Palm oil': {'Production': 44},
 'Soya beans': {'Area harvested': 68, 'Production': 68, 'Yield': 62}}


In [97]:
df_filtered = df.loc[
    (df.Item == 'Cattle') |
    ((df.Item == 'Palm oil') & (df.Element == 'Production' )) |
    ((df.Item == 'Cocoa beans') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Coffee, green') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Natural rubber in primary forms') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Soya beans') & (df.Element == 'Area harvested' )) 
]

In [98]:
# 2.71 Mha - area under cultivation of cocoa for 2021 by Kalischek 

df_filtered.loc[
    ((df_filtered.Area == 'Ghana') & 
     (df_filtered.Year == 2020) & 
     (df_filtered.Item == "Cocoa beans")), 'Value'] = 2710000


In [99]:
df_filtered

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,QCL,Crops and livestock products,4,Afghanistan,5111,Stocks,2111.000,Cattle,2000,2000,An,2900000.000,E,Estimated value,NaN
1,QCL,Crops and livestock products,4,Afghanistan,5111,Stocks,2111.000,Cattle,2020,2020,An,5326747.000,E,Estimated value,NaN
2,QCL,Crops and livestock products,8,Albania,5111,Stocks,2111.000,Cattle,2000,2000,An,728000.000,A,Official figure,NaN
3,QCL,Crops and livestock products,8,Albania,5111,Stocks,2111.000,Cattle,2020,2020,An,362583.000,A,Official figure,NaN
4,QCL,Crops and livestock products,8,Albania,5312,Area harvested,141.000,Soya beans,2000,2000,ha,400.000,A,Official figure,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2147,QCL,Crops and livestock products,716,Zimbabwe,5111,Stocks,2111.000,Cattle,2020,2020,An,5195154.000,A,Official figure,NaN
2148,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,1610.000,"Coffee, green",2000,2000,ha,6500.000,E,Estimated value,NaN
2151,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,1610.000,"Coffee, green",2020,2020,ha,2642.000,E,Estimated value,NaN
2154,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,141.000,Soya beans,2000,2000,ha,60650.000,A,Official figure,NaN


In [444]:
#tropical_countries_names = [*[value for key, value in country_name_mapping.items()], *tropical_countries['name'].tolist()]


In [100]:
df_filtered = df_filtered.loc[df_filtered.Area.isin(tropical_countries['name'])]

In [101]:
df_filtered.to_csv('../data/tmp/faostat_reports_2000_2020.csv', index=False)

In [102]:
df_filtered.loc[df_filtered.Area.str.startswith('V')]

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
2065,QCL,Crops and livestock products,548,Vanuatu,5111,Stocks,2111.000,Cattle,2000,2000,An,140000.000,E,Estimated value,NaN
2066,QCL,Crops and livestock products,548,Vanuatu,5111,Stocks,2111.000,Cattle,2020,2020,An,102776.000,I,Imputed value,NaN
2067,QCL,Crops and livestock products,548,Vanuatu,5312,Area harvested,1640.000,Cocoa beans,2000,2000,ha,3000.000,E,Estimated value,NaN
2070,QCL,Crops and livestock products,548,Vanuatu,5312,Area harvested,1640.000,Cocoa beans,2020,2020,ha,1447.000,I,Imputed value,NaN
2073,QCL,Crops and livestock products,548,Vanuatu,5312,Area harvested,1610.000,"Coffee, green",2000,2000,ha,40.000,E,Estimated value,NaN
2076,QCL,Crops and livestock products,548,Vanuatu,5312,Area harvested,1610.000,"Coffee, green",2020,2020,ha,59.000,E,Estimated value,NaN
2079,QCL,Crops and livestock products,862,Venezuela (Bolivarian Republic of),5111,Stocks,2111.000,Cattle,2000,2000,An,15164370.000,A,Official figure,NaN
2080,QCL,Crops and livestock products,862,Venezuela (Bolivarian Republic of),5111,Stocks,2111.000,Cattle,2020,2020,An,16055171.000,A,Official figure,NaN
2081,QCL,Crops and livestock products,862,Venezuela (Bolivarian Republic of),5312,Area harvested,1640.000,Cocoa beans,2000,2000,ha,56644.000,A,Official figure,NaN
2084,QCL,Crops and livestock products,862,Venezuela (Bolivarian Republic of),5312,Area harvested,1640.000,Cocoa beans,2020,2020,ha,80022.000,A,Official figure,NaN


In [103]:
def _get_commodity_expansion_rates(country, min_year, max_year=2020): #TODO: Max year should be 2022, this is temporary
        
    FAOSTAT_classes = ['Cattle', 'Palm oil', 'Soya beans', 'Cocoa beans', 'Coffee, green',
   'Natural rubber in primary forms']

    #TODO: this needs to be filtered so that for every commodity and year, there is only 1 element (either area, or production etc.)

    country_data = df_filtered.loc[df_filtered.Area == country]

    values = []

    for _class in FAOSTAT_classes:
        try:
            production_at_the_start = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(min_year)), 'Value' ].item()
            production_at_the_end = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(max_year)), 'Value' ].item()
            expansion = production_at_the_end - production_at_the_start
        except ValueError: #Signifies that one (or both) of the values are missing
            expansion = 0
        values.append(expansion)

    return values

In [104]:
vals = {}

for country in tropical_countries.name.unique():
    vals[country] = _get_commodity_expansion_rates(country, min_year=2000)

In [105]:
FAOSTAT_classes = ['Cattle', 'Palm oil', 'Soya beans', 'Cocoa beans', 'Coffee, green',
'Natural rubber in primary forms']

In [106]:
tmp_df = pd.DataFrame(vals).T
tmp_df.columns = FAOSTAT_classes

In [107]:
tmp_df

,Cattle,Palm oil,Soya beans,Cocoa beans,"Coffee, green",Natural rubber in primary forms
Fiji,-241699.000,0.000,0.000,-639.000,-8.000,0.000
United Republic of Tanzania,13551841.000,-740.000,36162.000,16484.000,116960.000,0.000
W. Sahara,0.000,0.000,0.000,0.000,0.000,0.000
United States of America,-4405700.000,0.000,4125820.000,0.000,0.000,0.000
Papua New Guinea,4668.000,344000.000,0.000,-2474.000,-43357.000,662.000
...,...,...,...,...,...,...
Somaliland,0.000,0.000,0.000,0.000,0.000,0.000
Uganda,8549833.000,0.000,-54000.000,55465.000,349885.000,0.000
Rwanda,876376.000,0.000,18945.000,0.000,-5147.000,0.000
Trinidad and Tobago,5575.000,0.000,0.000,-11278.000,765.000,0.000


In [108]:
areas = []
for idx, row in tmp_df.iterrows():
    _name = row.name
    area = tropical_countries.loc[tropical_countries['name'] == _name]['area'].item()
    areas.append(area)

tmp_df['country_area'] = areas

In [109]:
tmp_df
tmp_df.loc[tmp_df.index == "Viet Nam"]



,Cattle,Palm oil,Soya beans,Cocoa beans,"Coffee, green",Natural rubber in primary forms,country_area
Viet Nam,2102647.000,0.000,-82536.000,0.000,160663.000,497264.000,336019.374


In [110]:
def normalize_by_col(df, cols_to_normalize, norm_column):
    """For each column in df from cols_to_normalize, divide by the value in norm_column. Return normalized df"""
    df_normalized = df.copy()
    for col in cols_to_normalize:
        df_normalized[col] = df_normalized[col] / df_normalized[norm_column]
    return df_normalized

In [111]:
cols = ['Cattle', 'Palm oil', 'Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms',]

In [112]:
norm_df = normalize_by_col(tmp_df, cols, 'country_area')

In [113]:
norm_df

,Cattle,Palm oil,Soya beans,Cocoa beans,"Coffee, green",Natural rubber in primary forms,country_area
Fiji,-12.531,0.000,0.000,-0.033,-0.000,0.000,19287.604
United Republic of Tanzania,14.528,-0.001,0.039,0.018,0.125,0.000,932779.346
W. Sahara,0.000,0.000,0.000,0.000,0.000,0.000,96669.250
United States of America,-0.463,0.000,0.434,0.000,0.000,0.000,9509850.934
Papua New Guinea,0.010,0.741,0.000,-0.005,-0.093,0.001,464531.614
...,...,...,...,...,...,...,...
Somaliland,0.000,0.000,0.000,0.000,0.000,0.000,167473.218
Uganda,34.789,0.000,-0.220,0.226,1.424,0.000,245760.253
Rwanda,37.508,0.000,0.811,0.000,-0.220,0.000,23365.244
Trinidad and Tobago,0.720,0.000,0.000,-1.457,0.099,0.000,7738.923


### Calculate statistics from training set
- std and mean for cattle (unit: stocks), palm oil (unit: tons - production), rest (area harvested)

In [114]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)

In [115]:
### Doing it this way results in 0 values to become non-zero:
#y = (x - mean) / standard_deviation

'''stds = norm_df.std()
means = norm_df.mean()

harvested_area = norm_df[['Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms']].values.flatten()


mean = harvested_area.mean()
std = harvested_area.std()

normalization_training_dst = {
    'Cattle' : {
        'mean' : 2.541,
        'std' : 8.711
    },
    'Palm oil' : {
        'mean': 0.649,
        'std' : 3.125
    },
    
    'harvested_area' : {
        'mean' : 0.079,
        'std': 0.636
    }
    
}

norm_df['Cattle'] = (norm_df['Cattle'] - normalization_training_dst['Cattle']['mean'])/normalization_training_dst['Cattle']['std']
norm_df['Palm oil'] = (norm_df['Palm oil'] - normalization_training_dst['Palm oil']['mean'])/normalization_training_dst['Palm oil']['std']

for _cls in ['Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms']:
    
    norm_df[_cls] = (norm_df[_cls] - normalization_training_dst['harvested_area']['mean'])/normalization_training_dst['harvested_area']['std']

'''


"stds = norm_df.std()\nmeans = norm_df.mean()\n\nharvested_area = norm_df[['Soya beans', 'Cocoa beans', 'Coffee, green',\n       'Natural rubber in primary forms']].values.flatten()\n\n\nmean = harvested_area.mean()\nstd = harvested_area.std()\n\nnormalization_training_dst = {\n    'Cattle' : {\n        'mean' : 2.541,\n        'std' : 8.711\n    },\n    'Palm oil' : {\n        'mean': 0.649,\n        'std' : 3.125\n    },\n    \n    'harvested_area' : {\n        'mean' : 0.079,\n        'std': 0.636\n    }\n    \n}\n\nnorm_df['Cattle'] = (norm_df['Cattle'] - normalization_training_dst['Cattle']['mean'])/normalization_training_dst['Cattle']['std']\nnorm_df['Palm oil'] = (norm_df['Palm oil'] - normalization_training_dst['Palm oil']['mean'])/normalization_training_dst['Palm oil']['std']\n\nfor _cls in ['Soya beans', 'Cocoa beans', 'Coffee, green',\n       'Natural rubber in primary forms']:\n    \n    norm_df[_cls] = (norm_df[_cls] - normalization_training_dst['harvested_area']['mean'])/

### Normalization v2

just divide by max value for each of palm oil, cattle, area harvested

In [116]:
max_vals = norm_df.max()

In [117]:
norm_df['Cattle'] = norm_df['Cattle']/max_vals['Cattle']
norm_df['Palm oil'] = norm_df['Palm oil']/max_vals['Palm oil']


max_area = max_vals[['Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms']].max()

for _cls in ['Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms']:
    
    norm_df[_cls] = norm_df[_cls]/max_area


Index(['Fiji', 'United Republic of Tanzania', 'W. Sahara',
       'United States of America', 'Papua New Guinea', 'Indonesia',
       'Argentina', 'Chile', 'Democratic Republic of the Congo', 'Somalia',
       ...
       'Morocco', 'Egypt', 'Libya', 'Ethiopia', 'Djibouti', 'Somaliland',
       'Uganda', 'Rwanda', 'Trinidad and Tobago', 'South Sudan'],
      dtype='object', length=113)

In [120]:
norm_df['country'] = norm_df.index

In [122]:
norm_df.to_csv('../data/tmp/faostat_reports.csv', index=False)

## Read and explore the normalized data

In [2]:
norm_df = pd.read_csv('../data/tmp/faostat_reports.csv')

In [82]:
#norm_df['geometry'] = gpd.GeoSeries.from_wkt(norm_df['geometry'])

#gdf = gpd.GeoDataFrame(norm_df)

In [3]:
tmp = norm_df.set_index('country')

In [7]:
tmp

,Cattle,Palm oil,Soya beans,Cocoa beans,"Coffee, green",Natural rubber in primary forms,country_area
country,,,,,,,
Fiji,-0.334100,0.000000,0.000000,-0.004772,-0.000060,0.000000,1.928760e+04
United Republic of Tanzania,0.387346,-0.000032,0.005584,0.002545,0.018061,0.000000,9.327793e+05
W. Sahara,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.666925e+04
United States of America,-0.012352,0.000000,0.062491,0.000000,0.000000,0.000000,9.509851e+06
Papua New Guinea,0.000268,0.030182,0.000000,-0.000767,-0.013444,0.000205,4.645316e+05
...,...,...,...,...,...,...,...
Somaliland,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.674732e+05
Uganda,0.927525,0.000000,-0.031649,0.032508,0.205067,0.000000,2.457603e+05
Rwanda,1.000000,0.000000,0.116790,0.000000,-0.031730,0.000000,2.336524e+04


In [4]:
tmp['Cocoa beans'].sort_values(ascending=False)[:25]

country
Côte d'Ivoire                         1.000000
Ghana                                 0.715853
Sierra Leone                          0.175377
Liberia                               0.126110
Ecuador                               0.071527
Cameroon                              0.064903
Indonesia                             0.060100
Dominican Republic                    0.059033
Nigeria                               0.035406
Solomon Islands                       0.033441
Uganda                                0.032508
Guinea                                0.025280
Peru                                  0.015003
Colombia                              0.013159
Nicaragua                             0.011978
Philippines                           0.009474
Congo                                 0.008556
Timor-Leste                           0.007128
Costa Rica                            0.007110
Panama                                0.003973
India                                 0.003750
Venez

In [24]:
norm_df.to_csv('../data/tmp/faostat_reports.csv', index=False)

In [22]:
norm_df['country'] = tropical_countries['name'].reset_index(drop=True)

In [9]:
norm_df.loc[norm_df['country'] == "Malaysia"]



,Cattle,Palm oil,Soya beans,Cocoa beans,"Coffee, green",Natural rubber in primary forms,country_area,country
97,-0.003103,1.0,0.0,-0.029833,-0.011306,-0.124159,338220.757641,Malaysia


In [103]:
df_filtered['Flag Description'].unique()

array(['Official figure', 'Estimated value',
       'Missing value (data cannot exist, not applicable)',
       'Imputed value', 'Figure from international organizations'],
      dtype=object)

In [74]:
df = pd.read_csv("/home/jan/Documents/EPFL/project_2/data/labels/dataset_v4/labels_55k.csv")
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])

gdf = gpd.GeoDataFrame(df)

In [78]:
names = gdf.country.unique()

In [79]:
_dict = {}
test_countries = ['Ecuador', 'Malaysia', 'Thailand']
for name in names:
    if name not in test_countries:
        _dict[name] = "train"
    else:
        _dict[name] = "val"

In [80]:
import pprint
pprint.pprint(_dict)

{'Angola': 'train',
 'Australia': 'train',
 'Bahamas': 'train',
 'Bangladesh': 'train',
 'Belize': 'train',
 'Benin': 'train',
 'Bhutan': 'train',
 'Bolivia (Plurinational State of)': 'train',
 'Brazil': 'train',
 'Brunei Darussalam': 'train',
 'Burundi': 'train',
 'Cambodia': 'train',
 'Cameroon': 'train',
 'Central African Republic': 'train',
 'China': 'train',
 'Colombia': 'train',
 'Congo': 'train',
 'Costa Rica': 'train',
 'Cuba': 'train',
 "Côte d'Ivoire": 'train',
 'Democratic Republic of the Congo': 'train',
 'Dominican Republic': 'train',
 'Ecuador': 'val',
 'El Salvador': 'train',
 'Equatorial Guinea': 'train',
 'Ethiopia': 'train',
 'Fiji': 'train',
 'France': 'train',
 'Gabon': 'train',
 'Ghana': 'train',
 'Guatemala': 'train',
 'Guinea': 'train',
 'Guyana': 'train',
 'Haiti': 'train',
 'Honduras': 'train',
 'India': 'train',
 'Indonesia': 'train',
 'Jamaica': 'train',
 'Kenya': 'train',
 "Lao People's Democratic Republic": 'train',
 'Liberia': 'train',
 'Madagascar': 'trai